# Build a Knowledge Graph for Your QoS Design — the guided version

*A warm, step-by-step lab for network engineers. You bring the QoS. We bring the graph.*

Here is the thing about QoS that no one says plainly: **QoS never drops a link.** It
decides *who suffers when the pipe fills*. A congested link is a perfectly fine home for
your nightly backups and an absolute disaster for a voice call — and the only difference
between those two fates is **which class the traffic landed in** when the queue backed up.

That means the failure you care about here is **not** a binary link-down event. The link
is up. Packets are flowing. And yet a service is dying, quietly, inside a full queue —
because it was riding *best-effort* on a link with no room left. That is an **SLA breach**,
an *experience* failure, and it will never show up as a red interface in your NMS.

A knowledge graph makes the invisible visible. It makes explicit **which critical service is
riding best-effort, or riding a link with no QoS policy at all, the moment that link
congests** — so you find the SLA breach in a design review, before your users find it for you.

By the end of this notebook you will have built, from an empty page, a small graph that
answers the question no `show` command will:

> *"The WAN edge is congested — packets still flow. Which critical service is silently
> breaching its SLA right now?"*

and watched it print **`Payment API`** — a fact that lives in nobody's queue counters.

### First, calm the nerves: this is **not** machine learning
No training. No model. No embeddings. No AI guessing in the dark. A knowledge graph is
just **structured facts** (things, and the named links between them) plus **queries** that
walk those links. Everything is **deterministic and inspectable** — run it twice, get the
same answer, and you can point at every node that produced it. If you can read a
`show policy-map interface`, you can read every line in this lab.

### What you need
Nothing. This runs in **Google Colab** with zero setup, using plain **Python +
networkx + matplotlib** (both already installed in Colab). No database, no Docker, no
credentials. Press *Runtime -> Run all*, or run each cell in order with **Shift+Enter**.


## The whole vocabulary — five words, each one a QoS thing you already know

Before any code, here is the *entire* mental model. Everything after this is these five
ideas, repeated. You don't have to learn what any of them *mean* — only which QoS thing
each one maps onto.

| Graph word | Plain meaning | The QoS thing it already is |
|---|---|---|
| **node** | a thing | a link, a policy, a traffic class, a service, an SLA |
| **edge** | a named, directed link between two nodes | "this link applies that policy", "this service is classified as EF" |
| **label** | a node's *type* | `Link`, `QoSPolicy`, `TrafficClass`, `Service` — like "is this EF or best-effort?" |
| **property** | a fact stored *on* a node or edge | `state='congested'`, `priority=True`, `criticality='critical'` |
| **traversal** | walking edges to answer a question | "who's at risk on this full pipe?" — you'll do it by hand, on purpose |

That's it. Hold those five words and the rest is just your day job wearing a new hat.

One idea deserves a spotlight, because it is the whole trick of QoS: **the failure here is
not one fact — it is a *coincidence of two*.** "The pipe is full" (a `state` on the link) is
harmless on its own. "This service is best-effort" (an *edge* to a class) is harmless on its
own. The disaster is only born where the graph **joins them**: a *critical* service, that is
*best-effort*, on a link that is *congested*. No single counter holds that join. A graph does
nothing but hold joins. Keep an eye out for the moment those two harmless facts meet — that
is where an SLA quietly breaks, and where a diagram turns into something you can *query*.


## Setup — one import, one empty graph

**Theory (10 seconds).** `networkx` is a tiny Python library for building graphs in
memory. We will use a **`DiGraph`** — a *directed* graph, where every edge has a direction,
an arrow. Direction matters to us: `WAN-Edge APPLIES WAN-QoS` ("the link applies the
policy") is a different statement from the reverse, and `Payment API CLASSIFIED_AS
best-effort` only reads correctly one way. QoS is full of directional truth — traffic
flows *into* a class, a policy attaches *to* a link — so directed edges are the honest choice.

Run the next cell. It imports the tools and hands you an empty graph to fill.


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# Both libraries ship pre-installed in Colab -- nothing to pip install.
# A DiGraph is a *directed* graph: every edge is an arrow, from one node to another.
G = nx.DiGraph()

print('Empty graph ready:', G)
print('Nodes:', G.number_of_nodes(), ' Edges:', G.number_of_edges())

**Checkpoint.** You should see `Empty graph ready` and `Nodes: 0  Edges: 0`. That empty
`G` is your blank whiteboard. Every step from here just adds nodes and edges to it.


## Step 1 — Links: the pipes, and the one word that isn't "down"

**Theory.** The first thing worth writing down is the pipes themselves — the links whose
capacity is the whole reason QoS exists. Each link becomes a **node** with a **label** of
`Link` and two **properties**: its `capacity_mbps` and, the one that matters most, its
`state`.

And here is the distinction the whole lab hangs on, so let's nail it now: **`state` is not
`up`/`down`. It is `normal`/`congested`.** A congested link is *not* broken. It is up, it is
forwarding, `show interface` looks healthy — the packets are simply queuing, and some of
them are being dropped or delayed at the tail. Nothing pages. That is exactly why a QoS
failure is so dangerous: it hides inside a link that, by every binary check you have, is fine.

We'll model two links from the field: **WAN-Edge** — a small 100 Mbps WAN uplink that is
**congested** at peak (this is our incident) — and **MPLS-Core** — a fat, healthy 1 Gbps
core link in the **normal** state, our baseline for comparison.

Notice we are *not* dumping queue depths or per-flow telemetry in here. A node per link, a
*state* on it — not a node per dropped packet. We store the *shape* of the design; we'll
come back to that principle by name later.


In [ ]:
# add_node(name, **properties). The name is a unique handle; label & state are facts on it.
G.add_node('WAN-Edge',  label='Link', state='congested', capacity_mbps=100)   # the incident
G.add_node('MPLS-Core', label='Link', state='normal',    capacity_mbps=1000)  # healthy baseline

print(G.number_of_nodes(), 'links so far:', list(G.nodes))
print('Facts on WAN-Edge:', G.nodes['WAN-Edge'])

**Checkpoint.** Two nodes: `WAN-Edge` and `MPLS-Core`, and the facts on WAN-Edge show
`state='congested'` and `capacity_mbps=100`. Read that carefully: congested, **not** down.
You just wrote a full pipe into the graph as a plain fact you can query against — no edges
yet, just the pipes.


## Step 2 — The QoS policy and its classes: the priority ladder

**Theory.** A QoS policy is, at heart, a *ladder of classes* — the ordered decision about who
gets served first when the queue is full. **EF** (Expedited Forwarding) is the top rung, the
strict-priority queue voice rides. **AF41** is a guaranteed-bandwidth rung for critical data.
**best-effort** is the ground floor — no promise, served with whatever is left. When the pipe
fills, that ladder *is* the answer to "who suffers": the ground floor does.

In the graph, the policy is a **node** (`QoSPolicy`), each class is a **node**
(`TrafficClass`), and a `HAS_CLASS` **edge** ties them together. The single most important
**property** on a class is `priority` — `True` for EF and AF41 (they have a protected queue),
`False` for best-effort (it does not). That one boolean is what the blast-radius walk will
hinge on.

Then the pivotal wiring for *this* step: a link **`APPLIES`** a policy. `WAN-Edge APPLIES
WAN-QoS`. Note what we do **not** do — we leave **MPLS-Core with no policy at all**, and that
is fine: it is a lightly loaded 1 Gbps link with headroom to spare, and bolting QoS onto a
link that never congests is just complexity you'll maintain forever for no benefit. A link
with no policy is only a *risk* when that link **congests**. Hold that thought — it becomes a
second failure mode later.


In [ ]:
G.add_node('WAN-QoS', label='QoSPolicy', status='active')
G.add_node('EF',          label='TrafficClass', priority=True,  dscp='EF')
G.add_node('AF41',        label='TrafficClass', priority=True,  dscp='AF41')
G.add_node('best-effort', label='TrafficClass', priority=False, dscp='default')

G.add_edge('WAN-Edge', 'WAN-QoS', rel='APPLIES')          # the link applies the policy
for cls in ['EF', 'AF41', 'best-effort']:
    G.add_edge('WAN-QoS', cls, rel='HAS_CLASS')           # the policy holds each class

print('Policy applied to WAN-Edge:',
      [v for _, v, d in G.out_edges('WAN-Edge', data=True) if d['rel'] == 'APPLIES'])
for _, cls, d in G.out_edges('WAN-QoS', data=True):
    print(f'  class {cls:12} priority={G.nodes[cls]["priority"]}')

**Checkpoint.** `WAN-Edge` now applies `WAN-QoS`, and the policy holds three classes:
`EF` and `AF41` with `priority=True`, `best-effort` with `priority=False`. That is your
priority ladder, written down. `MPLS-Core` is deliberately still bare — a healthy link that
needs no policy. Next we decide *who rides which rung*, and that decision is the whole game.


## Step 3 — Classify the traffic (this is the pivotal step)

**Theory.** Here is the idea the whole lab pivots on. A policy with a beautiful priority
ladder protects *nobody* until you say **which service rides which rung**. That single
assignment — `Payment API CLASSIFIED_AS best-effort` versus `CLASSIFIED_AS AF41` — is the
*most load-bearing fact in the entire design*, and it is exactly the fact that lives nowhere
you can easily see: buried in an access-list, a class-map match, a marking someone set in 2019.

So we add two kinds of **edge**. First, `CLASSIFIED_AS`: each **Service** node (carrying a
`criticality` property) points at the class it lands in. Second, `TRAVERSES`: each service
points at the link it rides. Both are needed, because risk = *this service, in this class, on
this link.*

We wire three services onto the congested WAN-Edge, and we deliberately build all three
outcomes so you can feel the difference:

- **Voice** — `critical`, classified **EF**. The textbook-correct design: a critical, delay-
  sensitive service in a priority queue. When the pipe fills, voice is *protected*. Good.
- **Payment API** — `critical`, classified **best-effort**. The landmine. A revenue-critical
  service with **no priority queue**, riding the ground floor. When the pipe fills, it *suffers*.
- **Nightly Backup** — `low` criticality, classified **best-effort**. Also on the ground floor
  — and that is *exactly right*. A bulk backup is what best-effort is *for*; congestion
  slowing it down is a feature, not a fault.

Two services on the same best-effort rung, and only one is a problem — because best-effort is
correct for backups and catastrophic for payments. That contrast *is* QoS. The graph is about
to let us tell them apart automatically.


In [ ]:
G.add_node('Voice',          label='Service', criticality='critical')
G.add_node('Payment API',    label='Service', criticality='critical')
G.add_node('Nightly Backup', label='Service', criticality='low')

# who lands on which rung of the ladder -- the load-bearing decision
G.add_edge('Voice',          'EF',          rel='CLASSIFIED_AS')
G.add_edge('Payment API',    'best-effort', rel='CLASSIFIED_AS')   # <-- the landmine
G.add_edge('Nightly Backup', 'best-effort', rel='CLASSIFIED_AS')   # <-- and this one is fine

# and every one of them rides the congested WAN link
for svc in ['Voice', 'Payment API', 'Nightly Backup']:
    G.add_edge(svc, 'WAN-Edge', rel='TRAVERSES')

for svc in ['Voice', 'Payment API', 'Nightly Backup']:
    cls = [v for _, v, d in G.out_edges(svc, data=True) if d['rel'] == 'CLASSIFIED_AS'][0]
    print(f'{svc:16} criticality={G.nodes[svc]["criticality"]:9} class={cls}')

**Checkpoint.** Three services print with their class. Note the collision that matters:
`Payment API` (critical) and `Nightly Backup` (low) both sit in `best-effort` — same rung,
opposite meaning. `Voice` sits safely in `EF`. Every one of them `TRAVERSES` the congested
WAN-Edge. The facts are all in place now; nobody has *joined* them yet. That join is the
whole payoff, and it's two steps away.


## See it — draw the graph

**Theory.** Text is great during an incident, but to *understand* a design you want the
picture. We'll colour nodes by their **label** (links one colour, classes another, services
another), outline any **congested** link in red, and draw the dangerous edge — a *critical*
service classified into a *non-priority* class — as a dashed red arrow, so the landmine jumps
out. This is the same instinct as a design-review whiteboard, except this diagram is
generated *from the data*, so it can never drift out of sync with the truth.


In [ ]:
def draw(G, title='QoS knowledge graph'):
    colors = {'Link': '#3aa0ff', 'QoSPolicy': '#0f7f74', 'TrafficClass': '#8a6fb0',
              'Service': '#c8702a', 'SLA': '#b03a5b'}
    node_colors = [colors.get(G.nodes[n].get('label'), '#cccccc') for n in G]
    # congested links get a bold red outline; everything else a thin light one
    borders = ['#d34b3f' if G.nodes[n].get('state') == 'congested' else '#f4f4f4' for n in G]
    widths  = [3.0 if G.nodes[n].get('state') == 'congested' else 1.0 for n in G]
    pos = nx.spring_layout(G, seed=7)   # fixed seed => stable, repeatable layout

    plt.figure(figsize=(11, 7.5))
    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=1850, alpha=0.92,
                           edgecolors=borders, linewidths=widths)
    nx.draw_networkx_labels(G, pos, font_size=8)

    # the landmine: a CRITICAL service classified into a NON-priority class
    risky = [(u, v) for u, v, d in G.edges(data=True)
             if d.get('rel') == 'CLASSIFIED_AS'
             and G.nodes[u].get('criticality') == 'critical'
             and not G.nodes[v].get('priority')]
    other = [(u, v) for u, v, d in G.edges(data=True) if (u, v) not in risky]
    # curve every edge slightly so any reciprocal pair never overlaps
    nx.draw_networkx_edges(G, pos, edgelist=other, arrows=True, arrowsize=14,
                           edge_color='#8894a0', connectionstyle='arc3,rad=0.08')
    nx.draw_networkx_edges(G, pos, edgelist=risky, arrows=True, arrowsize=14,
                           edge_color='#d34b3f', width=2.0, style='dashed',
                           connectionstyle='arc3,rad=0.08')
    nx.draw_networkx_edge_labels(G, pos, font_size=6.5,
        edge_labels={(u, v): d.get('rel', '') for u, v, d in G.edges(data=True)})

    plt.axis('off'); plt.title(title); plt.tight_layout(); plt.show()

draw(G)

**Reading the picture.** You should see the links (blue) — with `WAN-Edge` ringed in **red**
because it is congested — the policy (teal), the three classes (purple), and the services
(orange) with `APPLIES`, `HAS_CLASS`, `CLASSIFIED_AS` and `TRAVERSES` arrows. One arrow is
**dashed red**: `Payment API -> best-effort`, a critical service on an unprotected rung. This
is still just a topology. It becomes a *knowledge* graph in the next step, when we add the
thing a QoS config has never carried: the **business promise** that this traffic is breaking.


## Step 4 — The SLA: the promise your queues never knew they were breaking

**Theory.** This is the node your policy-maps were never designed to hold, and the reason
the whole lab exists. A QoS config knows classes, markings, queues, and bandwidth percentages.
It has never once known that the Payment API owes the business **latency under 150 ms**, and
that the moment that traffic sits in a congested best-effort queue, *that promise is broken*
and money stops moving. The queue counter climbs; the meaning of the climb lives only in
someone's head.

So we add an **SLA** node — a business-facing promise with a `metric` and a `threshold` — and
wire it to the service with a `HAS_SLA` edge: *"this service owes this promise."* `Voice` owes
a jitter bound; `Payment API` owes a latency bound. One edge — and now a *queue-scheduling*
fact and a *business* fact are welded together in a place you can query. No `show` command
holds this link. It has only ever lived in tribal knowledge and a slide from a planning
meeting. You're about to make it a first-class, walkable fact — and give "at risk" a concrete
meaning: *which promise, breached by how much.*


In [ ]:
G.add_node('Voice-SLA',   label='SLA', metric='jitter',  threshold='30ms')
G.add_node('Payment-SLA', label='SLA', metric='latency', threshold='150ms')

G.add_edge('Voice',       'Voice-SLA',   rel='HAS_SLA')
G.add_edge('Payment API', 'Payment-SLA', rel='HAS_SLA')

print('Graph now:', G.number_of_nodes(), 'nodes,', G.number_of_edges(), 'edges')
for u, v, d in G.edges(data=True):
    if d['rel'] == 'HAS_SLA':
        print(f'  {u} promises {G.nodes[v]["metric"]} <= {G.nodes[v]["threshold"]}  ({v})')

draw(G, title='QoS knowledge graph -- now with the business SLAs')

**Checkpoint.** The graph has grown to **11 nodes, 12 edges**, including two pink `SLA` nodes.
In the redrawn picture you can now *trace with your finger*: congested WAN-Edge <- Payment API
-> best-effort (dashed red), and Payment API -> Payment-SLA. Every fact needed to spot the
breach is on the page. In the next step we make the computer trace it for you.


## Step 5 — The peak-hour question: blast radius as a traversal

**Theory.** Everything so far was setup for this. A **traversal** is just walking edges to
answer a question. Ours answers:

> *"For any link that is **congested**, which **critical** services are unprotected — either
> because they're riding **best-effort**, or because the link has **no QoS policy** at all?"*

The route the walk takes:

1. find a `Link` whose `state` is `congested`;
2. check whether an **active** `QoSPolicy` `APPLIES` to it (no policy at all is its own risk);
3. for each `Service` that `TRAVERSES` that link, keep only the `critical` ones;
4. a critical service is **at risk** if the link has no policy, **or** none of its
   `CLASSIFIED_AS` classes is a `priority` class (i.e. it is effectively best-effort);
5. report it, along with the SLA it is breaking.

Every hop is one edge. And notice the discipline in step 3: we filter on `criticality`. That
is what lets the graph do the thing a human does instinctively and a dumb alert cannot —
**ignore the Nightly Backup**. It's best-effort on the same congested link, but it's *supposed*
to be. The query surfaces the payment, stays silent about the backup. Run it.


In [ ]:
def blast_radius(G):
    """Critical services that are unprotected on a congested link."""
    at_risk = []
    for link, d in G.nodes(data=True):
        if d.get('label') != 'Link' or d.get('state') != 'congested':
            continue
        # 2) is an ACTIVE QoS policy actually applied to this link?
        has_policy = any(
            G.nodes[p].get('label') == 'QoSPolicy' and G.nodes[p].get('status') == 'active'
            for _, p, dd in G.out_edges(link, data=True) if dd.get('rel') == 'APPLIES')
        # 3) services that traverse this congested link
        for svc, _, de in G.in_edges(link, data=True):
            if de.get('rel') != 'TRAVERSES' or G.nodes[svc].get('criticality') != 'critical':
                continue
            # is this service in ANY priority class?
            in_priority = any(
                G.nodes[c].get('priority')
                for _, c, dc in G.out_edges(svc, data=True) if dc.get('rel') == 'CLASSIFIED_AS')
            # 4) unprotected = no policy on the link, OR not in a priority class
            if has_policy and in_priority:
                continue   # protected -- safe
            reason = 'link has no QoS policy applied' if not has_policy else 'critical traffic is best-effort'
            # 5) the SLA it breaches, if one is recorded
            sla = next((s for _, s, ds in G.out_edges(svc, data=True) if ds.get('rel') == 'HAS_SLA'), None)
            at_risk.append((link, svc, reason, sla))
    return at_risk

hits = blast_radius(G)
for link, svc, reason, sla in hits:
    tag = f'  (breaches {sla})' if sla else ''
    print(f'{link} CONGESTED  ->  AT RISK: {svc}   [{reason}]{tag}')
if not hits:
    print('nothing at risk')

Your queue counters only ever told you a buffer was filling — a number climbing on a graph.
Your knowledge graph just told you the **Payment API is breaching its 150 ms latency SLA**,
and showed its work: critical service, best-effort class, congested link. It stayed silent
about the Nightly Backup on that very same rung, because a backup on best-effort is working
*as designed*. You got that judgment because *you* welded the business promise onto the queue.
That is the entire pitch of a knowledge graph, and you just built it from an empty page.


## What-if: drain the pipe, then let it fill again

**Theory.** Because congestion lives on a node as a plain property, you can *change* it and
re-ask — running "what happens at peak?" (or "what if we add capacity?") on a **model**,
safely, with no one paged and no maintenance window. Flip WAN-Edge back to `normal`: the
blast radius clears, because step 1 of the walk finds no congested link. Flip it to
`congested` again: the Payment API breach returns. This is the humble beginning of
pre-incident planning — you can ask the graph what *would* break at next month's peak before
it does.


In [ ]:
G.nodes['WAN-Edge']['state'] = 'normal'      # the pipe drains -- headroom returns
print('After the pipe drains: ', blast_radius(G) or 'nothing at risk')

G.nodes['WAN-Edge']['state'] = 'congested'   # peak hour hits again
print('At peak again:         ', [svc for _, svc, *_ in blast_radius(G)])

**Checkpoint.** After the drain you should see `nothing at risk`; after re-congesting,
`['Payment API']` comes back. Same graph, same query — only the `state` on one link changed.
The answer *responded*. That is what makes it a model rather than a drawing.


## Your turn #1 — a second critical service on the same crowded rung

Real best-effort queues are crowded. Suppose **Video Conferencing** — also `critical`, also
delay-sensitive — got dropped into `best-effort` by the same tired classification. Add it,
classify it, route it over the congested `WAN-Edge`, and re-run `blast_radius`. You should
now see **two** critical services surface from the same full pipe — because one more edge of
true classification is all it takes.

Fill in the cell below (there's a `# TODO`), then run it. The solution follows if you want to
check.


In [ ]:
# TODO: add a 'Video Conferencing' Service (criticality='critical'),
#       classify it CLASSIFIED_AS 'best-effort', and route it TRAVERSES 'WAN-Edge'.

# G.add_node(...)
# G.add_edge(...)
# G.add_edge(...)

for link, svc, reason, sla in blast_radius(G):
    print(f'AT RISK: {svc}   [{reason}]')

<details><summary><b>Solution — click to reveal</b></summary>

```python
G.add_node('Video Conferencing', label='Service', criticality='critical')
G.add_edge('Video Conferencing', 'best-effort', rel='CLASSIFIED_AS')
G.add_edge('Video Conferencing', 'WAN-Edge',    rel='TRAVERSES')

for link, svc, reason, sla in blast_radius(G):
    print(f'AT RISK: {svc}   [{reason}]')
```

Now both `Payment API` **and** `Video Conferencing` come back from the one congested link.
The blast radius grew the instant you told the graph one more true thing — you didn't touch
the query at all.
</details>


In [ ]:
# (Solution, applied -- so the rest of the notebook carries both services.)
G.add_node('Video Conferencing', label='Service', criticality='critical')
G.add_edge('Video Conferencing', 'best-effort', rel='CLASSIFIED_AS')
G.add_edge('Video Conferencing', 'WAN-Edge',    rel='TRAVERSES')
print('Critical services at risk now:', sorted({svc for _, svc, *_ in blast_radius(G)}))

## Quiet reveal — you have been writing an *ontology*

Time to name the thing you've been doing. Every step, you made two very different kinds of
decision, and it's worth seeing the seam between them:

- *"A `Link` has a `state`. A `QoSPolicy` `HAS_CLASS` a `TrafficClass`. A `Service` is
  `CLASSIFIED_AS` a class and `TRAVERSES` a `Link`. A class has a `priority`."* — these are the
  **rules**: which node types exist, which edges are allowed, what shape a valid fact takes.
  That is the **schema**. Its fancy name is an **ontology** — and here's the friendly
  definition: *an ontology is the RFC of your graph, the agreed vocabulary written down as
  structure.* You already trust a Cisco QoS SRND to say what a valid class-map looks like; an
  ontology does the same job for your graph.

- *"This particular link is `WAN-Edge`, 100 Mbps, `congested`, and `Payment API` rides it as
  `best-effort`."* — these are the **instances**: the actual data.

A rule of thumb that keeps the two straight forever: **if it has a name, an interface, or a
number on it, it is data (an instance), never schema.** `TrafficClass` is schema;
`best-effort` is data. `CLASSIFIED_AS` is schema; "Payment API is best-effort" is data. Keep
the vocabulary small and stable; let the instances be many. That single discipline is the
difference between a graph you can grow for years and a swamp you abandon in a month.

*(For the curious: this lab's three risk shapes — a **defined policy not applied**, a
**critical application left best-effort**, and an **SLA with no protecting QoS** — faithfully
mirror the risk checks written into this project's real QoS ontology. You didn't build a toy;
you built a faithful miniature.)*


## A peek at the real thing (1/2) — the same links in Neo4j + Cypher

We used networkx so you could run everything with zero setup. But the *shapes* you built are
exactly what you'd build in a real graph database like **Neo4j**, whose query language is
**Cypher**. Cypher is worth a glance because it reads almost like the arrows we've been
drawing — `(node)-[:EDGE]->(node)`.

Here is **Steps 1-3** as Cypher. `MERGE` means "find this node or create it if missing" (so
re-running is safe); `SET` assigns properties — and notice we `MERGE` the *relationship* first,
then `SET` the fact on it, so the whole script is idempotent. This is *reference only* — you
don't run it here, it just shows the same idea in the grown-up tool:

```cypher
MERGE (link:WANLink {id: 'wan-edge'})
SET   link.state = 'congested', link.capacity_mbps = 100;

MERGE (pol:QoSPolicy {id: 'wan-qos'})
SET   pol.status = 'active';

MERGE (be:TrafficClass {id: 'best-effort'})
SET   be.priority = false;

// the policy is applied to the link -- a relationship, then a fact on it
MERGE (pol)-[r:APPLIED_TO]->(link)
SET   r.direction = 'output';

// the landmine: a critical service classified onto the ground floor
MERGE (svc:BusinessService {id: 'payment-api'})
SET   svc.criticality = 'critical';
MERGE (svc)-[c:CLASSIFIED_AS]->(be)
SET   c.marked = 'default';
```

See it? `(svc)-[:CLASSIFIED_AS]->(be)` is character-for-character the same statement as our
`G.add_edge('Payment API', 'best-effort', rel='CLASSIFIED_AS')`. Same node, same edge, same
fact-on-the-edge — one just happens to live in a database that scales to millions of nodes.

**One arrow reverses here, on purpose — and since we made such a fuss about direction,
it's worth owning.** In our Python we wrote `WAN-Edge APPLIES WAN-QoS` (link -> policy),
because "the link applies its policy" reads naturally left-to-right. The real ontology and
seed state it the other way: the **policy governs the link**, `(pol)-[:APPLIED_TO]->(link)`
(policy -> link). Both are valid framings of the same fact — we chose link -> policy in Python
for readability, and Neo4j convention runs policy -> link. Direction always matters; here two
conventions simply point the same truth in opposite directions.


## A peek at the real thing (2/2) — the peak-hour question in Cypher

Our `blast_radius` walk is a 20-line Python function. In Cypher, that same traversal is a
handful of lines, because in a graph database you **draw the shape you're looking for** and let
the engine find every match — no manual loops:

```cypher
MATCH (link:WANLink {state: 'congested'})<-[:TRAVERSES]-(svc:BusinessService {criticality: 'critical'})
OPTIONAL MATCH (pol:QoSPolicy {status: 'active'})-[:APPLIED_TO]->(link)
OPTIONAL MATCH (svc)-[:CLASSIFIED_AS]->(cls:TrafficClass {priority: true})
WITH svc, link, pol, cls
WHERE pol IS NULL OR cls IS NULL          // no policy on the link, OR not in a priority class
RETURN svc.name AS at_risk_service, link.name AS link,
       CASE WHEN pol IS NULL THEN 'link has no QoS policy applied'
            ELSE 'critical traffic is best-effort' END AS reason;
```

Read the first line as a sentence: *"match a congested WAN link with a critical service riding
it."* The two `OPTIONAL MATCH`es then ask "is it protected?", and the `WHERE` keeps only the
ones that aren't — the exact two failure modes you coded in Python. Run it against the real
QoS dataset in the companion Neo4j lab and `Payment API` comes back breaching its latency SLA,
right beside a congestion event with evidence attached. Different engine; identical thinking.
If you can read the Python above, you can read the Cypher — you already speak this language.


## Your turn #2 — the *other* way to be unprotected: a link with no policy

So far every risk came from a *misclassified* service. But remember MPLS-Core in Step 2 — a
link with **no QoS policy at all**. That's harmless while it has headroom. The moment such a
link **congests**, though, *every* service on it is unprotected — even one marked priority,
because a DSCP marking does nothing if no policy on the link acts on it. (In the real ontology
this is two named risks: a **defined policy not applied**, and an **SLA with no protecting QoS**.)

Prove it to yourself:

1. add a congested `Link` called `Branch-WAN` with **no policy applied**;
2. add a `critical` service `CRM App`, classify it `CLASSIFIED_AS 'AF41'` — a **priority**
   class! — and route it `TRAVERSES 'Branch-WAN'`;
3. re-run `blast_radius`. `CRM App` shows up **anyway** — its priority marking is worthless on a
   link with no policy to honour it.

This is the second failure mode, in your own hands: risk isn't only *wrong class*, it's also
*no policy*. Fill in the `# TODO`s, then run.


In [ ]:
# TODO 1: add Link 'Branch-WAN' (state='congested', capacity_mbps=50) with NO policy applied.
# TODO 2: add Service 'CRM App' (criticality='critical'),
#         classify it CLASSIFIED_AS 'AF41' (a priority class!),
#         and route it TRAVERSES 'Branch-WAN'.

# G.add_node(...); G.add_node(...)
# G.add_edge(...); G.add_edge(...)

for link, svc, reason, sla in blast_radius(G):
    print(f'{link}  ->  AT RISK: {svc}   [{reason}]')

<details><summary><b>Solution — click to reveal</b></summary>

```python
G.add_node('Branch-WAN', label='Link', state='congested', capacity_mbps=50)
G.add_node('CRM App',    label='Service', criticality='critical')
G.add_edge('CRM App', 'AF41',       rel='CLASSIFIED_AS')   # marked priority...
G.add_edge('CRM App', 'Branch-WAN', rel='TRAVERSES')       # ...but the link has no policy

for link, svc, reason, sla in blast_radius(G):
    print(f'{link}  ->  AT RISK: {svc}   [{reason}]')
```

`CRM App` surfaces on `Branch-WAN` with the reason *link has no QoS policy applied* — even
though you classified it into `AF41`, a priority class. The marking is real; it just has
nothing to act on. The query didn't need a new rule for this — the "no policy" branch was
there all along, waiting for a congested, policy-less link to exist. You told the graph the
truth, and the traversal did the rest.
</details>


In [ ]:
# (Solution, applied.) First the broken state, then the fix -- so the graph ends clean.
G.add_node('Branch-WAN', label='Link', state='congested', capacity_mbps=50)
G.add_node('CRM App',    label='Service', criticality='critical')
G.add_edge('CRM App', 'AF41',       rel='CLASSIFIED_AS')
G.add_edge('CRM App', 'Branch-WAN', rel='TRAVERSES')
print('Branch-WAN unprotected -> at risk:',
      [svc for link, svc, *_ in blast_radius(G) if link == 'Branch-WAN'])

# Fix it the way the ontology says: attach the approved policy to the link.
G.add_edge('Branch-WAN', 'WAN-QoS', rel='APPLIES')
print('After attaching the policy:   ',
      [svc for link, svc, *_ in blast_radius(G) if link == 'Branch-WAN'] or 'CRM App now protected')

## Make it yours — swap in *your* network

Now the moment it becomes your tool, not mine. Change the three values below to **one**
congested link, **one** critical service, and **one** SLA *you* actually run. We apply a
policy to the link but leave your service on `best-effort` — the single most common real-world
landmine — so your service shows up. Run it, and watch **your own service name** come back
from `blast_radius`, breaching **your own SLA** — proof that the machine you built understands
your network, not just the demo's.

Keep it to the smallest slice that answers one real question. You can always add one more node
tomorrow — that's the whole promise of a graph you can grow.


In [ ]:
# --- change these three values to your network ---
MY_LINK      = 'Datacenter-WAN'
MY_SERVICE   = 'Trading Gateway'
MY_SLA_LIMIT = '50ms latency'
# --------------------------------------------------

G.add_node(MY_LINK,    label='Link', state='congested', capacity_mbps=200)
G.add_edge(MY_LINK, 'WAN-QoS', rel='APPLIES')                 # the link HAS a policy...

G.add_node(MY_SERVICE, label='Service', criticality='critical')
G.add_edge(MY_SERVICE, 'best-effort', rel='CLASSIFIED_AS')    # ...but your service is best-effort
G.add_edge(MY_SERVICE, MY_LINK,       rel='TRAVERSES')

G.add_node(f'{MY_SERVICE}-SLA', label='SLA', metric='latency', threshold=MY_SLA_LIMIT)
G.add_edge(MY_SERVICE, f'{MY_SERVICE}-SLA', rel='HAS_SLA')

print(f'When {MY_LINK} congests, these critical services silently breach SLA:')
for link, svc, reason, sla in blast_radius(G):
    if link == MY_LINK:
        print(f'  AT RISK: {svc}   [{reason}]  (breaches {sla})')

**Checkpoint.** Your own service name — `Trading Gateway`, or whatever you typed — prints as
*at risk*, breaching your own SLA. That is the moment the tool stopped being a tutorial and
became yours. Every other service you run is just a few more lines away.


## The one rule that keeps this from becoming a swamp

You'll be tempted to model everything. Don't. Here is the discipline, in one line:

> **Model the nouns of your design review, not the numbers of your monitoring dashboard.**

Links, policies, classes, priorities, services, SLAs — the **nouns** you'd draw on a
whiteboard while arguing about a QoS design — those belong in the graph. Per-queue drop
counters, instantaneous buffer occupancy, NetFlow records, the interface's byte counters —
those are the **numbers**; leave them in the systems that already store them well, and let the
graph *reference* them as evidence when it needs to. The graph holds the *shape of the
dependency* — critical service, wrong class, full pipe, broken promise; your NMS holds the
firehose. Keep that line sharp and your graph stays queryable for years.

### Where to go next
- **The companion Neo4j lab** does this exact thing in a real graph database — same nodes,
  same edges, same peak-hour question — so the two Cypher peeks above become things you run,
  now with a `CongestionEvent`, an `SLAViolation`, and `Evidence` nodes carrying the proof.
- **Add the marking layer.** Model `DSCP`/`CoS` nodes and a `REQUIRES_MARKING` edge, and
  "where does the marking get rewritten between ingress and egress?" becomes one more traversal.
- **Add a protocol.** The shapes generalize: an OSPF adjacency is an edge with state; a BGP
  session is an edge with state; a redistribution policy is a node other things point at. The
  same five words carry all of it.


## You built a knowledge graph

Look back at the distance. You started with an empty page and five plain words. You added
links with a `state` that says `congested`, not `down` — a topology. Then you added the
priority ladder, classified the traffic, and welded on the one thing a QoS config never held —
the **business SLA** — and the topology became *knowledge*. Finally you asked it the question
no queue counter can answer, and it printed **Payment API**, named the promise it was breaking,
stayed silent about the backup it was right to ignore, and responded correctly every time you
changed the world underneath it.

The important idea was never QoS, and never networkx. It's this: **an SLA breach has a shape**
— a critical service, in the wrong class, on a full pipe, owing a promise — and once that
shape is a graph, "who suffers when the pipe fills?" stops being tribal knowledge and becomes
a traversal. A runbook is a hammer. A graph you can query is the whole toolbox.

QoS was never about dropping links. It was always about deciding who suffers. Now you can see
the decision — and find the one service suffering by accident — before your users ever feel it.
